
# PWD disclosure data, step by step

Introductory notebook for the `jollux-immigration-data` package: load a
small synthetic PWD fixture, run the public `ingest_pwd` function, and
inspect a few normalized fields. No network access, credentials,
database, or production data is needed.

Run from the repository root so the relative fixture path resolves:

```bash
python -m pip install -e .
jupyter notebook notebooks/pwd-intro.ipynb
```

See `notebooks/README.md` for the full environment notes.


In [1]:

import csv
from pathlib import Path

from jollux_immigration_data import ingest_pwd

fixture = Path("tests/fixtures/pwd.csv")

with fixture.open(encoding="utf-8") as handle:
    fixture_rows = sum(1 for _ in csv.reader(handle)) - 1

pwd_records = list(ingest_pwd(fixture))
print(f"fixture rows: {fixture_rows}")
print(f"records kept after PWD ingestion filters: {len(pwd_records)}")


fixture rows: 3
records kept after PWD ingestion filters: 1



Three fixture rows, one kept record. PWD ingestion (see the
[supported schema](../README.md#supported-schema) section of the main
README) keeps only `PERM` visa-class rows, drops `Withdrawn` statuses,
and requires an employer name and a wage-expiration date. The fixture's
other two rows fail those filters, so only `P-100` survives.


In [2]:

import json

record = pwd_records[0]
selected = {
    key: record[key]
    for key in (
        "case_number",
        "employer_name",
        "employer_normalized",
        "worksite_state",
        "worksite_location",
        "wage_level",
    )
}
print(json.dumps(selected, indent=2, ensure_ascii=False))


{
  "case_number": "P-100",
  "employer_name": "Example Analytics Inc.",
  "employer_normalized": "example analytics",
  "worksite_state": "TX",
  "worksite_location": "Austin, TX",
  "wage_level": "Level II"
}



A few things to notice in that output:

- `employer_name` keeps the source value; `employer_normalized` is the
  matching key you should store next to it.
- `worksite_state` is normalized to the USPS abbreviation;
  `worksite_location` joins city and state.
- `wage_level` maps common representations (`2`, `II`, `Level 2`) to
  `Level II`.

The full field list is in
[docs/supported-fields.md](../docs/supported-fields.md).
